In [22]:
# N7 bootstrap — defines everything from N6. Real code, no placeholders.
import os, sys, json, glob, zlib, subprocess, time
import numpy as np, cv2, torch, h5py
import torch.nn as nn, torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view
from google.colab import drive
drive.mount('/content/drive')

BASE  = '/content/drive/MyDrive/aruco_p2'
ART, WORK, CACHE = f'{BASE}/artifacts', '/content/work', f'{BASE}/archives'
os.makedirs(WORK, exist_ok=True); sys.path.insert(0, '/content')

REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/AVAuco/deeparuco.git', REPO], check=True)
sys.path.insert(0, REPO)
from impl.aruco import id_to_bits

# User-provided load_weights_safe function to handle HDF5 loading issues
import h5py

def load_weights_safe(h5_path):
    """Same output as load_weights, but walks only model_weights and skips broken links."""
    store = {}
    with h5py.File(h5_path, 'r') as f:
        root = f['model_weights'] if 'model_weights' in f else f
        def walk(g, prefix=''):
            for k in g.keys():
                try:
                    obj = g[k]
                except (KeyError, OSError):
                    continue                      # dangling link, e.g. input_1
                path = f'{prefix}/{k}'
                if isinstance(obj, h5py.Dataset):
                    store[path] = np.array(obj)
                elif isinstance(obj, h5py.Group):
                    walk(obj, path)
        walk(root)

    def pick(layer, suffix):
        hits = [k for k in store if f'/{layer}/' in k and k.endswith(suffix)]
        if len(hits) != 1:
            raise KeyError(f'{layer}:{suffix} -> {hits}\nall keys: {sorted(store)}')
        return store[hits[0]]

    w = {'c1_w': pick('conv2d','kernel:0'),   'c1_b': pick('conv2d','bias:0'),
         'p1_a': pick('p_re_lu','alpha:0'),
         'c2_w': pick('conv2d_1','kernel:0'), 'c2_b': pick('conv2d_1','bias:0'),
         'p2_a': pick('p_re_lu_1','alpha:0'),
         'c3_w': pick('conv2d_2','kernel:0'), 'c3_b': pick('conv2d_2','bias:0')}
    expect = {'c1_w':(5,5,1,8), 'c1_b':(8,), 'p1_a':(28,28,8),
              'c2_w':(3,3,8,4), 'c2_b':(4,), 'p2_a':(12,12,4),
              'c3_w':(1,1,4,1), 'c3_b':(1,)}
    for k, shape in expect.items():
        assert w[k].shape == shape, f'{k}: got {w[k].shape}, expected {shape}'
    return w

DEV_ = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
splits = json.load(open(f'{ART}/splits.json'))
DEV, HELDOUT, STRIDE = splits['dev'], splits['heldout'][0], splits['benchmark_stride']
Wk  = load_weights_safe(f'{REPO}/models/dec_new.h5') # Using the new safe loader
print('weights loaded:', {k: v.shape for k, v in Wk.items()})
CB  = np.stack([np.asarray(id_to_bits(i),np.uint8).reshape(6,6) for i in range(250)])
ROT = np.stack([np.rot90(CB,k,axes=(1,2)).reshape(250,36) for k in range(4)]).astype(np.float32)

SHADOW = f'{WORK}/shadow'; os.makedirs(SHADOW, exist_ok=True)
for i in range(1,7):
    if not os.path.isdir(f'{SHADOW}/video_{i}'):
        subprocess.run(['tar','-xzf',f'{CACHE}/video_{i}.tar.gz','-C',SHADOW], check=True)

r_int = lambda v: int(round(v))

def ordered_corners(xs, ys):
    cx, cy = np.mean(xs), np.mean(ys)
    idx = np.argsort(np.arctan2(xs-cx, ys-cy))
    return [v for i in idx for v in (xs[i], ys[i])]

def warp32(crop, c, t=32):
    dst = np.array([[0,0],[0,t-1],[t-1,t-1],[t-1,0]], np.float32)
    src = np.clip(np.array([[c[0]*63,c[1]*63],[c[2]*63,c[3]*63],
                            [c[4]*63,c[5]*63],[c[6]*63,c[7]*63]]),0,63).astype(np.float32)
    return cv2.warpPerspective(crop, cv2.getPerspectiveTransform(src,dst), (t,t))

def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict): return np.array([[p['x'],p['y']] for p in c], np.float32)
    return np.asarray(c, np.float32).reshape(4,2)

def cell_occlusion(m32):
    s = 32/8.0; out = np.zeros((6,6), bool)
    for r in range(6):
        for c in range(6):
            out[r,c] = m32[int((r+1)*s):int((r+2)*s), int((c+1)*s):int((c+2)*s)].mean() > 127.5
    return out

def prepare(img, mask, corners):
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None, None
    sl = (slice(r_int(mny), r_int(mxy)), slice(r_int(mnx), r_int(mxx)))
    crop = cv2.resize(img[sl], (64,64)).astype(np.float32)
    crop = (crop - crop.min())/(crop.max()-crop.min()+1e-9)
    nc = ordered_corners((x-mnx)/(mxx-mnx), (y-mny)/(mxy-mny))
    m32 = warp32(cv2.resize(mask[sl],(64,64)), nc) if mask.any() else np.zeros((32,32), np.uint8)
    return warp32(crop, nc), cell_occlusion(m32)

def occluder_box(img, corners, level, position, key, mid):
    if level <= 0: return None
    x, y = corners[:,0], corners[:,1]
    x0,x1,y0,y1 = x.min(),x.max(),y.min(),y.max(); w,h = x1-x0, y1-y0
    if position == 'edge':
        side = int(np.random.default_rng(
            zlib.crc32(f'occlusion|{level}|{key}|{mid}'.encode())).integers(0,4))
        box = [(x0,y0,x0+level*w,y1),(x1-level*w,y0,x1,y1),
               (x0,y0,x1,y0+level*h),(x0,y1-level*h,x1,y1)][side]
    else:
        cx, cy = (x0+x1)/2, (y0+y1)/2
        box = (cx-level*w/2, cy-level*h/2, cx+level*w/2, cy+level*h/2)
    a,b,c,d = [int(round(v)) for v in box]
    a,b = max(a,0),max(b,0); c,d = min(c,img.shape[1]),min(d,img.shape[0])
    return (a,b,c,d) if (c>a and d>b) else None

def apply_occluder(img, box, value):
    if box is None: return img, np.zeros(img.shape[:2], np.uint8)
    a,b,c,d = box
    out = img.copy(); mk = np.zeros(img.shape[:2], np.uint8)
    out[b:d,a:c] = int(value); mk[b:d,a:c] = 255
    return out, mk

def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j): out.append((f,j))
    return out

def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi',
                     sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))
def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)
def decode_torch(net, x8, chunk=8192):
    ps=[]
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            ps.append(torch.sigmoid(net(xb)[0]).cpu().numpy())
    return np.concatenate(ps)

class SpatialPReLU(nn.Module):
    def __init__(s,c,h,w): super().__init__(); s.a = nn.Parameter(torch.zeros(c,h,w))
    def forward(s,x): return torch.where(x>0, x, s.a.unsqueeze(0)*x)
class Decoder(nn.Module):
    def __init__(s, out_ch=1):
        super().__init__()
        s.c1=nn.Conv2d(1,8,5); s.p1=SpatialPReLU(8,28,28)
        s.c2=nn.Conv2d(8,4,3); s.p2=SpatialPReLU(4,12,12)
        s.c3=nn.Conv2d(4,out_ch,1); s.out_ch=out_ch
    def forward(s,x):
        x=F.max_pool2d(s.p1(s.c1(x)),2); x=F.max_pool2d(s.p2(s.c2(x)),2); x=s.c3(x)
        return (x[:,0], x[:,1]) if s.out_ch==2 else (x[:,0], None)

models = {}
for arm, oc, ck in [('B',1,f'{BASE}/ckpt_n3b/arm_B.pt'),
                    ('D',2,f'{BASE}/ckpt_n3b/arm_D.pt'),
                    ('E',1,f'{BASE}/ckpt_n6/arm_E.pt')]:
    assert os.path.exists(ck), f'{ck} missing'
    n = Decoder(oc).to(DEV_); n.load_state_dict(torch.load(ck, map_location=DEV_)); n.eval()
    models[arm] = n

assert len(pairs_for(DEV[0])) > 1000, f'only {len(pairs_for(DEV[0]))} pairs'
print(f'bootstrap OK | {len(pairs_for(DEV[0]))} pairs in {DEV[0]} | arms {sorted(models)}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
weights loaded: {'c1_w': (5, 5, 1, 8), 'c1_b': (8,), 'p1_a': (28, 28, 8), 'c2_w': (3, 3, 8, 4), 'c2_b': (4,), 'p2_a': (12, 12, 4), 'c3_w': (1, 1, 4, 1), 'c3_b': (1,)}
bootstrap OK | 1433 pairs in video_1 | arms ['B', 'D', 'E']


# N7 — Where do the errors land?

**Run this in the same session as N6.** It reuses `prepare`, `occluder_box`, `apply_occluder`,
`pairs_for`, `get_corners`, `cell_occlusion`, `models`, `decode_published`, `decode_torch`,
`CB`, `DEV`, `STRIDE`. Nothing is retrained.

### The question

N6 produced two unexplained curves:

| 18 occluded cells | black (0) | grey (127) | white (255) |
|---|---|---|---|
| A published | **44.5%** | 22.7% | **13.9%** |
| B grey-trained | **38.2%** | 83.2% | 72.5% |

Arm A's asymmetry is the clue. MIP36h12 is balanced (50.2% zeros), so a decoder that simply reads
covered cells at face value should fail *symmetrically*: imputing all-zeros and all-ones are
equally wrong on average. A gets 44.5% at black and 13.9% at white. Pure imputation cannot
produce that.

### Two mechanisms, one test

Split the Hamming distance by cell type:

| mechanism | errors on OCCLUDED cells | errors on VISIBLE cells |
|---|---|---|
| **imputation** — covered cells read at face value | ~half, flat in intensity | flat |
| **context disruption** — a white occluder destroys the marker's black border, which never occurs in training | varies | **rises with intensity** |

Spillover onto cells the decoder can still see is the only thing that explains the asymmetry.
This measures it directly.

Also reported: the mean predicted bit on occluded cells. Under imputation it should track occluder
intensity (≈0 for black, ≈1 for white).

## 1. Rebuild the sweep, keeping the occlusion masks
One severity, five intensities — about a fifth of N6's build.

In [23]:
# USER-PROVIDED GUARD: Start
for name in ['prepare','occluder_box','apply_occluder','pairs_for',
             'get_corners','cell_occlusion','decode_published','decode_torch']:
    assert name in globals(), f'{name} missing — run N6 cells 1-9 first, do NOT define placeholders'
assert 'models' in globals() and {'B','E'} <= set(models), 'arms B and E not loaded'
assert len(pairs_for(DEV[0])) > 1000, 'pairs_for is stubbed'
print('real N6 environment confirmed')
# USER-PROVIDED GUARD: End

import numpy as np, cv2, json, time
import os

INTENS7 = [0, 64, 127, 192, 255]
LEVEL7  = 0.50          # ~18 occluded cells at the edge, per N6

S7 = {v: {'x':[], 'id':[], 'mask':[], 'frame':[]} for v in INTENS7}
t0 = time.time()
for vid in DEV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            box = occluder_box(img, cn, LEVEL7, 'edge', key, mid)
            for v in INTENS7:
                oi, om = apply_occluder(img, box, v)
                x32, occ = prepare(oi, om, cn)
                if x32 is None: continue
                S7[v]['x'].append((x32*255).astype(np.uint8)); S7[v]['id'].append(mid)
                S7[v]['mask'].append(occ); S7[v]['frame'].append(key)
    print(f'  {vid} ({time.time()-t0:.0f}s)')
for v in INTENS7:
    for k in S7[v]: S7[v][k] = np.array(S7[v][k])
# USER-PROVIDED ASSERT: Start
assert len(S7[0]['id']) > 5000, 'S7[0]["id"] is too small, likely stubbed'
# USER-PROVIDED ASSERT: End
print(f'\n{len(S7[0]["id"])} markers per intensity')
print(f'mean occluded cells: {S7[0]["mask"].sum(axis=(1,2)).mean():.1f} of 36')
# geometry is identical across intensities by construction
assert all(np.array_equal(S7[INTENS7[0]]['mask'], S7[v]['mask']) for v in INTENS7[1:]), \
    'masks differ across intensity — geometry is not controlled'
print('masks identical across intensities: geometry controlled')

real N6 environment confirmed
  video_1 (19s)
  video_2 (32s)
  video_3 (48s)
  video_4 (64s)
  video_5 (77s)

12276 markers per intensity
mean occluded cells: 18.0 of 36
masks identical across intensities: geometry controlled


## 2. Decompose the errors

`bits_true` is the true codeword in the observed orientation, found by taking the rotation that
minimises total error — the same convention the decoder search uses.

In [24]:
def true_bits_oriented(P, ids):
    """Return the true codeword rotated to match the observed orientation."""
    obs = (P > 0.5).astype(np.uint8)
    out = np.empty_like(obs)
    for n in range(len(P)):
        ref = CB[ids[n]]
        best, bk = 99, 0
        for k in range(4):
            h = int(np.sum(np.rot90(obs[n], k) != ref))
            if h < best: best, bk = h, k
        out[n] = np.rot90(ref, -bk)          # bring the reference into observed frame
    return out

RES7 = {}
for v in INTENS7:
    s = S7[v]
    RES7[v] = {}
    for arm in ('A','B','E'):
        P = decode_published(s['x']) if arm == 'A' else decode_torch(models[arm], s['x'])
        T = true_bits_oriented(P, s['id'])
        obs = (P > 0.5).astype(np.uint8)
        err = (obs != T)
        M = s['mask']
        RES7[v][arm] = {
            'err_occ':  err[M].mean(),
            'err_vis':  err[~M].mean(),
            'bit_occ':  P[M].mean(),
            'bit_vis':  P[~M].mean(),
            'conf_occ': (np.abs(P-0.5)*2)[M].mean(),
        }
    print(f'intensity {v:>3} done')

intensity   0 done
intensity  64 done
intensity 127 done
intensity 192 done
intensity 255 done


## 3. Results

In [25]:
for arm in ('A','B','E'):
    print(f'\n--- arm {arm} ---')
    print(f"{'occluder':>9}{'err OCCLUDED':>15}{'err VISIBLE':>14}{'mean bit occ':>15}{'conf occ':>11}")
    print('-'*64)
    for v in INTENS7:
        r = RES7[v][arm]
        print(f'{v:>9}{r["err_occ"]:>15.3f}{r["err_vis"]:>14.3f}{r["bit_occ"]:>15.3f}{r["conf_occ"]:>11.3f}')
    ev = [RES7[v][arm]['err_vis'] for v in INTENS7]
    eo = [RES7[v][arm]['err_occ'] for v in INTENS7]
    print(f'  visible-cell error range : {min(ev):.3f} -> {max(ev):.3f}  (spread {max(ev)-min(ev):.3f})')
    print(f'  occluded-cell error range: {min(eo):.3f} -> {max(eo):.3f}  (spread {max(eo)-min(eo):.3f})')


--- arm A ---
 occluder   err OCCLUDED   err VISIBLE   mean bit occ   conf occ
----------------------------------------------------------------
        0          0.499         0.085          0.063      0.915
       64          0.430         0.131          0.923      0.987
      127          0.419         0.176          0.966      0.992
      192          0.412         0.205          0.978      0.994
      255          0.408         0.223          0.980      0.994
  visible-cell error range : 0.085 -> 0.223  (spread 0.138)
  occluded-cell error range: 0.408 -> 0.499  (spread 0.092)

--- arm B ---
 occluder   err OCCLUDED   err VISIBLE   mean bit occ   conf occ
----------------------------------------------------------------
        0          0.496         0.129          0.105      0.794
       64          0.449         0.061          0.661      0.473
      127          0.471         0.082          0.514      0.206
      192          0.482         0.100          0.490      0.191
     

## 4. Which mechanism?

In [26]:
print('IMPUTATION predicts: mean bit on occluded cells tracks occluder intensity,')
print('                    visible-cell error flat, occluded-cell error ~0.5 throughout.')
print('CONTEXT DISRUPTION predicts: visible-cell error rises with intensity.\n')

for arm in ('A','B','E'):
    ev = np.array([RES7[v][arm]['err_vis'] for v in INTENS7])
    bo = np.array([RES7[v][arm]['bit_occ'] for v in INTENS7])
    x  = np.array(INTENS7, float)/255.0
    spread_vis = ev.max() - ev.min()
    corr_bit   = float(np.corrcoef(x, bo)[0,1])
    corr_vis   = float(np.corrcoef(x, ev)[0,1])
    print(f'arm {arm}:')
    print(f'  corr(intensity, mean bit on occluded cells) = {corr_bit:+.3f}   '
          f'{"imputation confirmed" if corr_bit > 0.8 else "NOT simple imputation"}')
    print(f'  corr(intensity, VISIBLE-cell error)         = {corr_vis:+.3f}')
    print(f'  visible-cell error spread                   = {spread_vis:.3f}   '
          f'{"SPILLOVER" if spread_vis > 0.03 else "no spillover"}')
    print()

sp = max(np.ptp([RES7[v][a]['err_vis'] for v in INTENS7]) for a in ('A','B','E'))
if sp > 0.05:
    print('CONTEXT DISRUPTION. Occluder intensity corrupts cells the decoder can still see.')
    print('The failure is not local imputation — it is global. That is a mechanism, and it')
    print('explains the asymmetry a balanced dictionary cannot.')
elif sp > 0.02:
    print('PARTIAL spillover. Report both components; neither alone explains the curves.')
else:
    print('NO spillover: errors stay on occluded cells. The asymmetry must come from')
    print('somewhere else — check whether the black BORDER cells are being covered,')
    print('since an edge occluder reaches the border before it reaches any data cell.')

IMPUTATION predicts: mean bit on occluded cells tracks occluder intensity,
                    visible-cell error flat, occluded-cell error ~0.5 throughout.
CONTEXT DISRUPTION predicts: visible-cell error rises with intensity.

arm A:
  corr(intensity, mean bit on occluded cells) = +0.742   NOT simple imputation
  corr(intensity, VISIBLE-cell error)         = +0.985
  visible-cell error spread                   = 0.138   SPILLOVER

arm B:
  corr(intensity, mean bit on occluded cells) = +0.461   NOT simple imputation
  corr(intensity, VISIBLE-cell error)         = +0.056
  visible-cell error spread                   = 0.068   SPILLOVER

arm E:
  corr(intensity, mean bit on occluded cells) = +0.656   NOT simple imputation
  corr(intensity, VISIBLE-cell error)         = +0.134
  visible-cell error spread                   = 0.050   SPILLOVER

CONTEXT DISRUPTION. Occluder intensity corrupts cells the decoder can still see.
The failure is not local imputation — it is global. That is a mecha

## 5. The border check

An edge occluder crosses the marker's 1-cell black border before reaching any data cell. At
intensity 0 the border stays black and the marker looks intact; at 255 the border is destroyed.
That is a marker-specific asymmetry a balanced codeword table cannot produce, and it is the most
likely explanation if section 4 shows no spillover.

This measures the border directly.

In [27]:
def border_stats(x8, mask_frac=True):
    """Mean intensity of the border ring (the 28 outer cells of the 8x8 grid)."""
    s = 32/8.0
    out = np.zeros(len(x8), np.float32)
    for n, img in enumerate(x8):
        vals = []
        for r in range(8):
            for c in range(8):
                if r in (0,7) or c in (0,7):
                    vals.append(img[int(r*s):int((r+1)*s), int(c*s):int((c+1)*s)].mean())
        out[n] = np.mean(vals)
    return out

print(f"{'occluder':>9}{'border mean':>13}{'border std':>12}   (border is black in an intact marker)")
print('-'*52)
for v in INTENS7:
    b = border_stats(S7[v]['x'][:2000])
    print(f'{v:>9}{b.mean():>13.1f}{b.std():>12.1f}')

print('\nIf border mean rises steeply with occluder intensity while accuracy falls,')
print('the mechanism is border destruction: the decoder never sees a marker without a')
print('dark border during training, so a bright occluder puts the whole crop out of')
print('distribution rather than just hiding cells.')

json.dump({'intensities': INTENS7, 'level': LEVEL7,
           'results': {f'{v}_{a}': {k: float(x) for k,x in RES7[v][a].items()}
                       for v in INTENS7 for a in ('A','B','E')},
           'viewpoints': DEV, 'heldout_untouched': HELDOUT},
          open(f'{ART}/n7_error_localisation.json','w'), indent=1)
print(f'\nsaved -> {ART}/n7_error_localisation.json')

 occluder  border mean  border std   (border is black in an intact marker)
----------------------------------------------------
        0         27.3        18.5
       64         78.0        42.4
      127        100.2        27.6
      192        119.6        13.7
      255        135.4        13.0

If border mean rises steeply with occluder intensity while accuracy falls,
the mechanism is border destruction: the decoder never sees a marker without a
dark border during training, so a bright occluder puts the whole crop out of
distribution rather than just hiding cells.

saved -> /content/drive/MyDrive/aruco_p2/artifacts/n7_error_localisation.json
